# 🛢️ Taller: Importación y Extracción de Datos

**Inteligencia Artificial con Aplicaciones en Economía I**

**Universidad Externado de Colombia — Facultad de Economía**

**Integrantes del grupo:**
- Daniel Peña
- Santiago González
- Sebastián Mendoza




## ⚙️ Instalación de librerías necesarias

Estas instalaciones son necesarias la primera vez que se corre el notebook en Google Colab.

In [1]:
# Para leer y escribir archivos de Excel
!pip install xlrd openpyxl -q

# Para trabajar con datos tabulares
!pip install pandas -q

# Para conectarnos a la API del Banco Mundial de forma sencilla
!pip install wbdata -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 316.5/316.5 kB 4.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
moviepy 1.0.3 requires decorator<5.0,>=4.0.2, but you have decorator 5.3.1 which is incompatible.


## 📚 Importar librerías

In [2]:
import os
import datetime
import requests
import numpy as np
import pandas as pd
import wbdata

from google.colab import drive

## 🖥️ Mejorar la visualización de los DataFrames

In [3]:
# Que muestre todas las columnas
pd.options.display.max_columns = None
# Mostrar los float con dos decimales
pd.options.display.float_format = '{:,.2f}'.format

## 🔎 Función auxiliar para explorar cada conjunto de datos

Como el taller pide revisar exactamente lo mismo para **cada** conjunto de datos (puntos 1 al 5), creamos
una función que hace todas las revisiones de una sola vez, para no repetir el mismo código 5 veces:

- Primeras 5 filas
- Últimas 10 filas
- 3 filas aleatorias
- Filas 2 a 5, solo la segunda columna
- `len()` → número de filas
- `.shape` → filas y columnas
- `.columns` → nombres de columnas
- `.dtypes` → tipo de dato de cada columna
- `.info()` → resumen general (filas, columnas, tipos, nulos)

In [4]:
def explorar_dataset(df, nombre="Dataset"):
    """Aplica el checklist de exploración del taller a un DataFrame."""
    print(f"\n{'='*70}")
    print(f"📦 EXPLORACIÓN DE: {nombre}")
    print(f"{'='*70}")

    print("\n➡️ Primeras 5 filas (.head()):")
    display(df.head())

    print("\n➡️ Últimas 10 filas (.tail(10)):")
    display(df.tail(10))

    print("\n➡️ 3 filas aleatorias (.sample(3)):")
    display(df.sample(min(3, len(df))))

    print("\n➡️ Filas 2 a 5, solo la segunda columna (.iloc[1:5, [1]]):")
    display(df.iloc[1:5, [1]])

    print(f"\n➡️ Número de filas (len()): {len(df)}")

    print(f"\n➡️ Filas y columnas (.shape): {df.shape}")

    print("\n➡️ Nombre de las columnas (.columns):")
    print(df.columns.tolist())

    print("\n➡️ Tipo de dato de cada columna (.dtypes):")
    display(df.dtypes)

    print("\n➡️ Información general (.info()):")
    df.info()

---
# 1️. Base de países del WEO (FMI) desde Google Drive

**Pasos previos (se hacen UNA vez, fuera del notebook):**

1. Ir a la página del *World Economic Outlook Database* del FMI: https://www.imf.org/en/Publications/WEO/weo-database (o el nuevo portal https://data.imf.org/en/datasets/IMF.RES:WEO ).
2. Descargar la base completa "by countries" (todos los países, todos los años) en formato Excel/tabular.
3. Subir ese archivo descargado a Google Drive (por ejemplo, en una carpeta `Taller_Datos`).
4. Ajustar en la celda siguiente la ruta y el nombre exacto del archivo que subieron.

In [6]:
# Conectarse a Google Drive
from google.colab import drive, files
drive.mount('/content/drive')

Mounted at /content/drive


In [7]:
# Para establecer el directorio de los archivos
os.chdir("/content/drive/MyDrive/taller ia importacion y exportacion de datos")

In [9]:
# Archivo Excel
df_WEO_mun = pd.read_excel("weoapr2025all.xlsx")
df_WEO_mun.head()

,WEO Country Code,ISO,WEO Subject Code,Country,Subject Descriptor,Subject Notes,Units,Scale,Country/Series-specific Notes,1980,1981,1982,1983,1984,1985,1986,1987,1988,1989,1990,1991,1992,1993,1994,1995,1996,1997,1998,1999,2000,2001,2002,2003,2004,2005,2006,2007,2008,2009,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025,2026,2027,2028,2029,2030,Estimates Start After
0,512,AFG,NGDP_R,Afghanistan,"Gross domestic product, constant prices",Expressed in billions of national currency uni...,National currency,Billions,Source: National Statistics Office Latest actu...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,453484,492903,496209,554910,584658,662650,688247,829924,899956,958266,"1,092.118","1,154.178","1,185.306","1,197.012","1,222.917","1,255.288","1,270.216","1,319.902","1,288.869","1,101.445","1,032.712","1,056.123",NaN,NaN,NaN,NaN,NaN,NaN,NaN,"2,024.00"
1,512,AFG,NGDP_RPCH,Afghanistan,"Gross domestic product, constant prices",Annual percentages of constant price GDP are y...,Percent change,Units,"See notes for: Gross domestic product, consta...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,8692,0.671,11830,5361,13340,3863,20585,8438,6479,13968,5683,2697,0.988,2164,2647,1189,3912,-2351,-14542,-6240,2267,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"2,024.00"
2,512,AFG,NGDP,Afghanistan,"Gross domestic product, current prices",Expressed in billions of national currency uni...,National currency,Billions,Source: National Statistics Office Latest actu...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,178756,220013,246210,304926,345817,427495,517509,607227,711759,836222,"1,033.591","1,116.827","1,183.039","1,226.570","1,222.917","1,285.460","1,327.690","1,469.596","1,547.289","1,251.172","1,283.442","1,350.910",NaN,NaN,NaN,NaN,NaN,NaN,NaN,"2,024.00"
3,512,AFG,NGDPD,Afghanistan,"Gross domestic product, current prices",Values are based upon GDP in national currency...,U.S. dollars,Billions,"See notes for: Gross domestic product, curren...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4367,4553,5146,6167,6925,8556,10297,12066,15325,17890,20293,20170,20616,20057,18020,18883,18336,18876,20136,14278,14501,17248,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"2,024.00"
4,512,AFG,PPPGDP,Afghanistan,"Gross domestic product, current prices",These data form the basis for the country weig...,Purchasing power parity; international dollars,Billions,"See notes for: Gross domestic product, curren...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,22742,25206,26058,30054,32642,37999,40227,48807,53568,58216,67583,72639,75898,77358,79784,83362,89369,97801,100898,85768,86149,91272,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"2,024.00"


In [10]:
# 👉 AJUSTAR: ruta de la carpeta en su Drive donde subieron el archivo del WEO
ruta_carpeta = '/content/drive/MyDrive/taller ia importacion y exportacion de datos'
os.chdir(ruta_carpeta)

# 👉 AJUSTAR: nombre exacto del archivo del WEO que subieron a Drive
nombre_archivo_weo = 'weoapr2025all.xlsx'

try:
    df_weo = pd.read_excel(nombre_archivo_weo)
except Exception:
    # Algunos archivos del WEO en realidad son texto separado por tabulaciones
    df_weo = pd.read_csv(nombre_archivo_weo, sep='\t', encoding='latin1')

df_weo.head()

,WEO Country Code,ISO,WEO Subject Code,Country,Subject Descriptor,Subject Notes,Units,Scale,Country/Series-specific Notes,1980,1981,1982,1983,1984,1985,1986,1987,1988,1989,1990,1991,1992,1993,1994,1995,1996,1997,1998,1999,2000,2001,2002,2003,2004,2005,2006,2007,2008,2009,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025,2026,2027,2028,2029,2030,Estimates Start After
0,512,AFG,NGDP_R,Afghanistan,"Gross domestic product, constant prices",Expressed in billions of national currency uni...,National currency,Billions,Source: National Statistics Office Latest actu...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,453484,492903,496209,554910,584658,662650,688247,829924,899956,958266,"1,092.118","1,154.178","1,185.306","1,197.012","1,222.917","1,255.288","1,270.216","1,319.902","1,288.869","1,101.445","1,032.712","1,056.123",NaN,NaN,NaN,NaN,NaN,NaN,NaN,"2,024.00"
1,512,AFG,NGDP_RPCH,Afghanistan,"Gross domestic product, constant prices",Annual percentages of constant price GDP are y...,Percent change,Units,"See notes for: Gross domestic product, consta...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,8692,0.671,11830,5361,13340,3863,20585,8438,6479,13968,5683,2697,0.988,2164,2647,1189,3912,-2351,-14542,-6240,2267,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"2,024.00"
2,512,AFG,NGDP,Afghanistan,"Gross domestic product, current prices",Expressed in billions of national currency uni...,National currency,Billions,Source: National Statistics Office Latest actu...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,178756,220013,246210,304926,345817,427495,517509,607227,711759,836222,"1,033.591","1,116.827","1,183.039","1,226.570","1,222.917","1,285.460","1,327.690","1,469.596","1,547.289","1,251.172","1,283.442","1,350.910",NaN,NaN,NaN,NaN,NaN,NaN,NaN,"2,024.00"
3,512,AFG,NGDPD,Afghanistan,"Gross domestic product, current prices",Values are based upon GDP in national currency...,U.S. dollars,Billions,"See notes for: Gross domestic product, curren...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4367,4553,5146,6167,6925,8556,10297,12066,15325,17890,20293,20170,20616,20057,18020,18883,18336,18876,20136,14278,14501,17248,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"2,024.00"
4,512,AFG,PPPGDP,Afghanistan,"Gross domestic product, current prices",These data form the basis for the country weig...,Purchasing power parity; international dollars,Billions,"See notes for: Gross domestic product, curren...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,22742,25206,26058,30054,32642,37999,40227,48807,53568,58216,67583,72639,75898,77358,79784,83362,89369,97801,100898,85768,86149,91272,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"2,024.00"


In [11]:
explorar_dataset(df_weo, "Base de países - WEO (FMI)")


📦 EXPLORACIÓN DE: Base de países - WEO (FMI)

➡️ Primeras 5 filas (.head()):


,WEO Country Code,ISO,WEO Subject Code,Country,Subject Descriptor,Subject Notes,Units,Scale,Country/Series-specific Notes,1980,1981,1982,1983,1984,1985,1986,1987,1988,1989,1990,1991,1992,1993,1994,1995,1996,1997,1998,1999,2000,2001,2002,2003,2004,2005,2006,2007,2008,2009,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025,2026,2027,2028,2029,2030,Estimates Start After
0,512,AFG,NGDP_R,Afghanistan,"Gross domestic product, constant prices",Expressed in billions of national currency uni...,National currency,Billions,Source: National Statistics Office Latest actu...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,453484,492903,496209,554910,584658,662650,688247,829924,899956,958266,"1,092.118","1,154.178","1,185.306","1,197.012","1,222.917","1,255.288","1,270.216","1,319.902","1,288.869","1,101.445","1,032.712","1,056.123",NaN,NaN,NaN,NaN,NaN,NaN,NaN,"2,024.00"
1,512,AFG,NGDP_RPCH,Afghanistan,"Gross domestic product, constant prices",Annual percentages of constant price GDP are y...,Percent change,Units,"See notes for: Gross domestic product, consta...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,8692,0.671,11830,5361,13340,3863,20585,8438,6479,13968,5683,2697,0.988,2164,2647,1189,3912,-2351,-14542,-6240,2267,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"2,024.00"
2,512,AFG,NGDP,Afghanistan,"Gross domestic product, current prices",Expressed in billions of national currency uni...,National currency,Billions,Source: National Statistics Office Latest actu...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,178756,220013,246210,304926,345817,427495,517509,607227,711759,836222,"1,033.591","1,116.827","1,183.039","1,226.570","1,222.917","1,285.460","1,327.690","1,469.596","1,547.289","1,251.172","1,283.442","1,350.910",NaN,NaN,NaN,NaN,NaN,NaN,NaN,"2,024.00"
3,512,AFG,NGDPD,Afghanistan,"Gross domestic product, current prices",Values are based upon GDP in national currency...,U.S. dollars,Billions,"See notes for: Gross domestic product, curren...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4367,4553,5146,6167,6925,8556,10297,12066,15325,17890,20293,20170,20616,20057,18020,18883,18336,18876,20136,14278,14501,17248,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"2,024.00"
4,512,AFG,PPPGDP,Afghanistan,"Gross domestic product, current prices",These data form the basis for the country weig...,Purchasing power parity; international dollars,Billions,"See notes for: Gross domestic product, curren...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,22742,25206,26058,30054,32642,37999,40227,48807,53568,58216,67583,72639,75898,77358,79784,83362,89369,97801,100898,85768,86149,91272,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"2,024.00"



➡️ Últimas 10 filas (.tail(10)):


,WEO Country Code,ISO,WEO Subject Code,Country,Subject Descriptor,Subject Notes,Units,Scale,Country/Series-specific Notes,1980,1981,1982,1983,1984,1985,1986,1987,1988,1989,1990,1991,1992,1993,1994,1995,1996,1997,1998,1999,2000,2001,2002,2003,2004,2005,2006,2007,2008,2009,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025,2026,2027,2028,2029,2030,Estimates Start After
8616,698,ZWE,GGXONLB_NGDP,Zimbabwe,General government primary net lending/borrowing,Primary net lending/borrowing is net lending (...,Percent of GDP,Units,See notes for: General government primary net...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-2599,0.147,-0.911,0.290,-0.010,0.790,-2174,0.056,-0.360,-0.324,-1149,-6004,-9434,-4573,-2013,0.090,-2573,-4695,-5198,-1096,0.537,1529,1948,2028,2108,2108,"2,024.00"
8617,698,ZWE,GGXWDN,Zimbabwe,General government net debt,Net debt is calculated as gross debt minus fin...,National currency,Billions,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8618,698,ZWE,GGXWDN_NGDP,Zimbabwe,General government net debt,Net debt is calculated as gross debt minus fin...,Percent of GDP,Units,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8619,698,ZWE,GGXWDG,Zimbabwe,General government gross debt,Gross debt consists of all liabilities that re...,National currency,Billions,Source: Ministry of Finance or Treasury Latest...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.001,0.001,0.001,0.002,0.002,0.002,0.002,0.003,0.003,0.003,0.004,0.004,0.006,0.007,0.070,0.466,0.743,4935,51682,557793,673344,739670,786902,862793,930773,993329,"2,024.00"
8620,698,ZWE,GGXWDG_NGDP,Zimbabwe,General government gross debt,Gross debt consists of all liabilities that re...,Percent of GDP,Units,See notes for: General government gross debt ...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,33062,39438,44671,61078,58710,47578,42858,38379,37048,42312,48029,49908,68935,48125,82337,84466,58172,99536,96586,94587,58568,56051,53294,52315,50485,48312,"2,024.00"
8621,698,ZWE,NGDP_FY,Zimbabwe,Gross domestic product corresponding to fiscal...,Gross domestic product corresponding to fiscal...,National currency,Billions,Source: Ministry of Finance or Treasury Latest...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.004,0.004,0.003,0.003,0.003,0.003,0.004,0.004,0.005,0.005,0.005,0.004,0.004,0.004,0.004,0.004,0.003,0.003,0.003,0.004,0.005,0.006,0.007,0.008,0.008,0.008,0.008,0.009,0.015,0.085,0.552,1277,4958,53509,589716,"1,149.685","1,319.643","1,476.529","1,649.212","1,843.656","2,056.061","2,024.00"
8622,698,ZWE,BCA,Zimbabwe,Current account balance,Current account is all transactions other than...,U.S. dollars,Billions,Source: Reserve Bank of Zimbabwe and Ministry ...,-0.301,-0.674,-0.748,-0.504,-0.171,-0.153,-0.051,0.000,0.050,-0.079,-0.257,-0.547,-0.842,-0.311,-0.318,-0.369,-0.180,-0.801,-0.159,0.244,0.322,0.365,0.249,-0.069,-0.128,-0.087,0.154,0.298,-0.138,-0.730,-1655,-2750,-2278,-2649,-2334,-1597,-0.697,-0.271,-1380,0.920,0.678,0.348,0.305,0.135,0.505,1143,1210,1198,1226,1120,0.996,"2,023.00"
8623,698,ZWE,BCA_NGDPD,Zimbabwe,Current account balance,Current account is all transactions other than...,Percent of GDP,Units,"See notes for: Gross domestic product, curren...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-2534,-5789,-10804,-4109,-3995,-4466,-1779,-7717,-1312,2070,2838,3248,2323,-0.720,-1353,-0.958,1887,3829,-2061,-7557,-13747,-19497,-13308,-13874,-11971,-7998,-3394,-1230,-3737,3536,2524,0.967,0.935,0.384,1435,2994,3063,2927,2895,2554,2196,"2,023.00"
8624,NaN,NaN,Na


➡️ 3 filas aleatorias (.sample(3)):


...


➡️ Filas 2 a 5, solo la segunda columna (.iloc[1:5, [1]]):


,ISO
1,AFG
2,AFG
3,AFG
4,AFG



➡️ Número de filas (len()): 8626

➡️ Filas y columnas (.shape): (8626, 61)

➡️ Nombre de las columnas (.columns):
['WEO Country Code', 'ISO', 'WEO Subject Code', 'Country', 'Subject Descriptor', 'Subject Notes', 'Units', 'Scale', 'Country/Series-specific Notes', 1980, 1981, 1982, 1983, 1984, 1985, 1986, 1987, 1988, 1989, 1990, 1991, 1992, 1993, 1994, 1995, 1996, 1997, 1998, 1999, 2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025, 2026, 2027, 2028, 2029, 2030, 'Estimates Start After']

➡️ Tipo de dato de cada columna (.dtypes):


,0
WEO Country Code,object
ISO,object
WEO Subject Code,object
Country,object
Subject Descriptor,object
...,...
2027,object
2028,object
2029,object
2030,object



➡️ Información general (.info()):
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8626 entries, 0 to 8625
Data columns (total 61 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   WEO Country Code               8625 non-null   object 
 1   ISO                            8624 non-null   object 
 2   WEO Subject Code               8624 non-null   object 
 3   Country                        8624 non-null   object 
 4   Subject Descriptor             8624 non-null   object 
 5   Subject Notes                  8624 non-null   object 
 6   Units                          8624 non-null   object 
 7   Scale                          8624 non-null   object 
 8   Country/Series-specific Notes  7648 non-null   object 
 9   1980                           3861 non-null   object 
 10  1981                           3995 non-null   object 
 11  1982                           4039 non-null   object 
 12  1983         

---
# 2️. PIB real (2010-2023): Estados Unidos, Alemania, Canadá, España, Francia e Italia

Usamos la API del Banco Mundial a través de la librería `wbdata`.

- Indicador: `NY.GDP.MKTP.KD` → PIB (US$ a precios constantes de 2015), es decir, el **PIB real**.
- Países: Estados Unidos (USA), Alemania (DEU), Canadá (CAN), España (ESP), Francia (FRA), Italia (ITA).
- Periodo: 2010-2023.

In [12]:
indicador_pib_real = {"NY.GDP.MKTP.KD": "PIB real (US$ constantes de 2015)"}

paises_punto2 = ["USA", "DEU", "CAN", "ESP", "FRA", "ITA"]

fecha_inicio = datetime.datetime(2010, 1, 1)
fecha_fin = datetime.datetime(2023, 1, 1)

df_pib_real = wbdata.get_dataframe(indicador_pib_real, country=paises_punto2,
                                    date=(fecha_inicio, fecha_fin))

# Pasamos el índice (país, fecha) a columnas normales para que sea más fácil de explorar
df_pib_real = df_pib_real.reset_index()
df_pib_real.head()

,country,date,PIB real (US$ constantes de 2015)
0,Canada,2023,"1,822,053,687,169.68"
1,Canada,2022,"1,787,149,136,135.45"
2,Canada,2021,"1,706,998,394,665.32"
3,Canada,2020,"1,611,127,783,961.78"
4,Canada,2019,"1,696,606,794,629.52"


In [13]:
explorar_dataset(df_pib_real, "PIB real 2010-2023 (EE.UU., Alemania, Canadá, España, Francia, Italia)")


📦 EXPLORACIÓN DE: PIB real 2010-2023 (EE.UU., Alemania, Canadá, España, Francia, Italia)

➡️ Primeras 5 filas (.head()):


,country,date,PIB real (US$ constantes de 2015)
0,Canada,2023,"1,822,053,687,169.68"
1,Canada,2022,"1,787,149,136,135.45"
2,Canada,2021,"1,706,998,394,665.32"
3,Canada,2020,"1,611,127,783,961.78"
4,Canada,2019,"1,696,606,794,629.52"



➡️ Últimas 10 filas (.tail(10)):


,country,date,PIB real (US$ constantes de 2015)
74,United States,2019,"20,159,639,089,819.00"
75,United States,2018,"19,651,869,117,636.50"
76,United States,2017,"19,085,691,123,385.50"
77,United States,2016,"18,627,887,993,795.50"
78,United States,2015,"18,295,019,000,000.00"
79,United States,2014,"17,771,549,056,393.40"
80,United States,2013,"17,334,068,402,849.70"
81,United States,2012,"16,974,575,729,644.70"
82,United States,2011,"16,594,704,135,237.80"
83,United States,2010,"16,339,094,225,608.80"



➡️ 3 filas aleatorias (.sample(3)):


...


➡️ Filas 2 a 5, solo la segunda columna (.iloc[1:5, [1]]):


,date
1,2022
2,2021
3,2020
4,2019



➡️ Número de filas (len()): 84

➡️ Filas y columnas (.shape): (84, 3)

➡️ Nombre de las columnas (.columns):
['country', 'date', 'PIB real (US$ constantes de 2015)']

➡️ Tipo de dato de cada columna (.dtypes):


,0
country,object
date,object
PIB real (US$ constantes de 2015),float64



➡️ Información general (.info()):
<class 'wbdata.client.DataFrame'>
RangeIndex: 84 entries, 0 to 83
Data columns (total 3 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   country                            84 non-null     object 
 1   date                               84 non-null     object 
 2   PIB real (US$ constantes de 2015)  84 non-null     float64
dtypes: float64(1), object(2)
memory usage: 2.1+ KB


---
# 3️. Brasil (2010-2023): crecimiento del PIB, PIB per cápita PPA, inflación, desempleo y población

Indicadores del Banco Mundial:

| Indicador | Código |
|---|---|
| Crecimiento del PIB (% anual) | `NY.GDP.MKTP.KD.ZG` |
| PIB per cápita, PPA (US$ internacionales corrientes) | `NY.GDP.PCAP.PP.CD` |
| Tasa de inflación, precios al consumidor (% anual) | `FP.CPI.TOTL.ZG` |
| Tasa de desempleo (% de la fuerza laboral total, estimación OIT) | `SL.UEM.TOTL.ZS` |
| Población total | `SP.POP.TOTL` |

In [14]:
indicadores_brasil = {
    "NY.GDP.MKTP.KD.ZG": "Crecimiento del PIB (%)",
    "NY.GDP.PCAP.PP.CD": "PIB per cápita PPA (US$ internacionales)",
    "FP.CPI.TOTL.ZG": "Tasa de inflación (%)",
    "SL.UEM.TOTL.ZS": "Tasa de desempleo (%)",
    "SP.POP.TOTL": "Población total",
}

df_brasil = wbdata.get_dataframe(indicadores_brasil, country="BRA",
                                  date=(fecha_inicio, fecha_fin))

df_brasil = df_brasil.reset_index()
df_brasil.head()

,date,Crecimiento del PIB (%),PIB per cápita PPA (US$ internacionales),Tasa de inflación (%),Tasa de desempleo (%),Población total
0,2023,3.24,"21,193.07",4.59,7.95,"211,140,729.00"
1,2022,3.02,"19,873.70",9.28,9.23,"210,306,415.00"
2,2021,4.76,"18,075.71",8.30,13.16,"209,550,294.00"
3,2020,-3.28,"16,101.62",3.21,13.70,"208,660,842.00"
4,2019,1.22,"16,069.84",3.73,11.94,"207,455,459.00"


In [15]:
explorar_dataset(df_brasil, "Brasil 2010-2023 (crecimiento PIB, PIB per cápita PPA, inflación, desempleo, población)")


📦 EXPLORACIÓN DE: Brasil 2010-2023 (crecimiento PIB, PIB per cápita PPA, inflación, desempleo, población)

➡️ Primeras 5 filas (.head()):


,date,Crecimiento del PIB (%),PIB per cápita PPA (US$ internacionales),Tasa de inflación (%),Tasa de desempleo (%),Población total
0,2023,3.24,"21,193.07",4.59,7.95,"211,140,729.00"
1,2022,3.02,"19,873.70",9.28,9.23,"210,306,415.00"
2,2021,4.76,"18,075.71",8.30,13.16,"209,550,294.00"
3,2020,-3.28,"16,101.62",3.21,13.70,"208,660,842.00"
4,2019,1.22,"16,069.84",3.73,11.94,"207,455,459.00"



➡️ Últimas 10 filas (.tail(10)):


,date,Crecimiento del PIB (%),PIB per cápita PPA (US$ internacionales),Tasa de inflación (%),Tasa de desempleo (%),Población total
4,2019,1.22,"16,069.84",3.73,11.94,"207,455,459.00"
5,2018,1.78,"15,463.74",3.66,12.33,"206,107,261.00"
6,2017,1.32,"14,559.05",3.45,12.79,"204,703,445.00"
7,2016,-3.28,"14,309.21",8.74,11.58,"203,218,114.00"
8,2015,-3.55,"14,821.44",9.03,8.54,"201,675,532.00"
9,2014,0.50,"15,827.31",6.33,6.75,"200,085,127.00"
10,2013,3.00,"15,722.33",6.20,7.07,"198,478,299.00"
11,2012,1.92,"15,198.09",5.40,7.25,"196,876,111.00"
12,2011,3.97,"15,211.79",6.64,7.58,"195,284,734.00"
13,2010,7.53,"14,451.72",5.04,8.42,"193,701,929.00"



➡️ 3 filas aleatorias (.sample(3)):


...


➡️ Filas 2 a 5, solo la segunda columna (.iloc[1:5, [1]]):


,Crecimiento del PIB (%)
1,3.02
2,4.76
3,-3.28
4,1.22



➡️ Número de filas (len()): 14

➡️ Filas y columnas (.shape): (14, 6)

➡️ Nombre de las columnas (.columns):
['date', 'Crecimiento del PIB (%)', 'PIB per cápita PPA (US$ internacionales)', 'Tasa de inflación (%)', 'Tasa de desempleo (%)', 'Población total']

➡️ Tipo de dato de cada columna (.dtypes):


,0
date,object
Crecimiento del PIB (%),float64
PIB per cápita PPA (US$ internacionales),float64
Tasa de inflación (%),float64
Tasa de desempleo (%),float64
Población total,float64



➡️ Información general (.info()):
<class 'wbdata.client.DataFrame'>
RangeIndex: 14 entries, 0 to 13
Data columns (total 6 columns):
 #   Column                                    Non-Null Count  Dtype  
---  ------                                    --------------  -----  
 0   date                                      14 non-null     object 
 1   Crecimiento del PIB (%)                   14 non-null     float64
 2   PIB per cápita PPA (US$ internacionales)  14 non-null     float64
 3   Tasa de inflación (%)                     14 non-null     float64
 4   Tasa de desempleo (%)                     14 non-null     float64
 5   Población total                           14 non-null     float64
dtypes: float64(5), object(1)
memory usage: 804.0+ bytes


---
# 4️. Dos conjuntos de datos del portal de Datos Abiertos de Colombia (datos.gov.co)

El portal https://www.datos.gov.co usa la plataforma **Socrata**. Para cada conjunto de datos, entramos a la página
del dataset → botón **Exportar** → **API de acceso**, y copiamos el endpoint que termina en `.json`.

Elegimos dos conjuntos de datos que nos llamaron la atención:

1. **Tasa de Cambio Representativa del Mercado (TRM)** — tasa de cambio oficial peso/dólar publicada por la
   Superintendencia Financiera.
2. **Códigos Postales Nacionales** — códigos postales de los municipios de Colombia.

⚠️ Si alguno de los dos endpoints ya no existe o cambió, simplemente busquen otro dataset en datos.gov.co que les
llame la atención, entren a **Exportar → API de acceso**, copien el link `.json` y reemplácenlo en la celda
correspondiente; el resto del código funciona igual.

### 4.1 Tasa de Cambio Representativa del Mercado (TRM)

In [16]:
url_trm = "https://www.datos.gov.co/resource/32sa-8pi3.json?$limit=5000"

response = requests.get(url_trm)
data_trm = response.json()

df_trm = pd.DataFrame(data_trm)
df_trm.head()

,valor,unidad,vigenciadesde,vigenciahasta
0,3118.24,COP,2026-08-27T00:00:00.000,2026-08-27T00:00:00.000
1,3081.67,COP,2026-08-26T00:00:00.000,2026-08-26T00:00:00.000
2,3056.51,COP,2026-08-25T00:00:00.000,2026-08-25T00:00:00.000
3,3048.12,COP,2026-08-22T00:00:00.000,2026-08-24T00:00:00.000
4,3062.96,COP,2026-08-21T00:00:00.000,2026-08-21T00:00:00.000


In [17]:
explorar_dataset(df_trm, "TRM - Datos Abiertos Colombia")


📦 EXPLORACIÓN DE: TRM - Datos Abiertos Colombia

➡️ Primeras 5 filas (.head()):


,valor,unidad,vigenciadesde,vigenciahasta
0,3118.24,COP,2026-08-27T00:00:00.000,2026-08-27T00:00:00.000
1,3081.67,COP,2026-08-26T00:00:00.000,2026-08-26T00:00:00.000
2,3056.51,COP,2026-08-25T00:00:00.000,2026-08-25T00:00:00.000
3,3048.12,COP,2026-08-22T00:00:00.000,2026-08-24T00:00:00.000
4,3062.96,COP,2026-08-21T00:00:00.000,2026-08-21T00:00:00.000



➡️ Últimas 10 filas (.tail(10)):


,valor,unidad,vigenciadesde,vigenciahasta
4990,2308.42,COP,2005-08-02T00:00:00.000,2005-08-02T00:00:00.000
4991,2308.49,COP,2005-07-30T00:00:00.000,2005-08-01T00:00:00.000
4992,2311.43,COP,2005-07-29T00:00:00.000,2005-07-29T00:00:00.000
4993,2313.84,COP,2005-07-28T00:00:00.000,2005-07-28T00:00:00.000
4994,2316.93,COP,2005-07-27T00:00:00.000,2005-07-27T00:00:00.000
4995,2315.39,COP,2005-07-26T00:00:00.000,2005-07-26T00:00:00.000
4996,2316.42,COP,2005-07-23T00:00:00.000,2005-07-25T00:00:00.000
4997,2312.82,COP,2005-07-22T00:00:00.000,2005-07-22T00:00:00.000
4998,2312.73,COP,2005-07-20T00:00:00.000,2005-07-21T00:00:00.000
4999,2320.26,COP,2005-07-19T00:00:00.000,2005-07-19T00:00:00.000



➡️ 3 filas aleatorias (.sample(3)):


...


➡️ Filas 2 a 5, solo la segunda columna (.iloc[1:5, [1]]):


,unidad
1,COP
2,COP
3,COP
4,COP



➡️ Número de filas (len()): 5000

➡️ Filas y columnas (.shape): (5000, 4)

➡️ Nombre de las columnas (.columns):
['valor', 'unidad', 'vigenciadesde', 'vigenciahasta']

➡️ Tipo de dato de cada columna (.dtypes):


,0
valor,object
unidad,object
vigenciadesde,object
vigenciahasta,object



➡️ Información general (.info()):
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 4 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   valor          5000 non-null   object
 1   unidad         5000 non-null   object
 2   vigenciadesde  5000 non-null   object
 3   vigenciahasta  5000 non-null   object
dtypes: object(4)
memory usage: 156.4+ KB


### 4.2 Códigos Postales Nacionales

In [18]:
url_postal = "https://www.datos.gov.co/resource/ixig-z8b5.json?$limit=5000"

response = requests.get(url_postal)
data_postal = response.json()

df_postal = pd.DataFrame(data_postal)
df_postal.head()

,noid,codigo_departamento,nombre_departamento,codigo_municipio,nombre_municipio,zona_postal,codigo_postal,limite_norte,limite_sur,limite_este,limite_oeste,tipo,barrios_contenidos_en_el,veredas_contenidas_en_el
0,877,15,BOYACA,15621,RONDON,1.534,153.42,LIMITE URBANO,LIMITE URBANO,LIMITE URBANO,LIMITE URBANO,Urbano,Sin Informacion de Barrios,Sin Informacion de Veredas
1,1.765,25,CUNDINAMARCA,25592,QUEBRADANEGRA,2.534,253.427,MUNICIPIO UTICA,MUNICIPIO VILLETA,MUNICIPIO NIMAIMA,MUNICIPIO GUADUAS,Rural,Sin Informacion de Barrios,Cune- GALETA- HATO- LA ESPERANZA- LA FLORID...
2,228,5,ANTIOQUIA,5002,ABEJORRAL,550,55.03,LIMITE URBANO,LIMITE URBANO,LIMITE URBANO,LIMITE URBANO,Urbano,Sin Informacion de Barrios,Alto de Letras
3,3.54,88,ARCHIPIELAGO DE SAN ANDRES,88001,SAN ANDRES,8.8,880.007,LIMITE URBANO,SAN LUIS,MAR CARIBE,MAR CARIBE,Rural,Sin Informacion de Barrios,Sin Informacion de Veredas
4,1.963,41,HUILA,41013,AGRADO,4.14,414.04,LIMITE URBANO,LIMITE URBANO,KR 1,KR 11,Urbano,Sin Informacion de Barrios,Bajo Buena Vista- Cabecera Municipal- La Galda


In [19]:
explorar_dataset(df_postal, "Códigos Postales Nacionales - Datos Abiertos Colombia")


📦 EXPLORACIÓN DE: Códigos Postales Nacionales - Datos Abiertos Colombia

➡️ Primeras 5 filas (.head()):


,noid,codigo_departamento,nombre_departamento,codigo_municipio,nombre_municipio,zona_postal,codigo_postal,limite_norte,limite_sur,limite_este,limite_oeste,tipo,barrios_contenidos_en_el,veredas_contenidas_en_el
0,877,15,BOYACA,15621,RONDON,1.534,153.42,LIMITE URBANO,LIMITE URBANO,LIMITE URBANO,LIMITE URBANO,Urbano,Sin Informacion de Barrios,Sin Informacion de Veredas
1,1.765,25,CUNDINAMARCA,25592,QUEBRADANEGRA,2.534,253.427,MUNICIPIO UTICA,MUNICIPIO VILLETA,MUNICIPIO NIMAIMA,MUNICIPIO GUADUAS,Rural,Sin Informacion de Barrios,Cune- GALETA- HATO- LA ESPERANZA- LA FLORID...
2,228,5,ANTIOQUIA,5002,ABEJORRAL,550,55.03,LIMITE URBANO,LIMITE URBANO,LIMITE URBANO,LIMITE URBANO,Urbano,Sin Informacion de Barrios,Alto de Letras
3,3.54,88,ARCHIPIELAGO DE SAN ANDRES,88001,SAN ANDRES,8.8,880.007,LIMITE URBANO,SAN LUIS,MAR CARIBE,MAR CARIBE,Rural,Sin Informacion de Barrios,Sin Informacion de Veredas
4,1.963,41,HUILA,41013,AGRADO,4.14,414.04,LIMITE URBANO,LIMITE URBANO,KR 1,KR 11,Urbano,Sin Informacion de Barrios,Bajo Buena Vista- Cabecera Municipal- La Galda



➡️ Últimas 10 filas (.tail(10)):


,noid,codigo_departamento,nombre_departamento,codigo_municipio,nombre_municipio,zona_postal,codigo_postal,limite_norte,limite_sur,limite_este,limite_oeste,tipo,barrios_contenidos_en_el,veredas_contenidas_en_el
3671,2.432,52,NARIÑO,52320,GUAITARILLA,5.255,525.508,MUNICIPIO ANCUYÁ,MUNICIPIO IMUES,Q/DA. GRANADILLO Y RELIEVE M.,M. TUQUE.- PROVIDEN. Y SAMANIEGO,Rural,Sin Informacion de Barrios,Guaramuez
3672,1.406,23,CORDOBA,23001,MONTERIA,2.3,230.029,VÍA MONTERIA-PUERTO REY,Q. TENERIFE- EL COCUELO Y CAÑO TRES BO.,RÍO SINÚ,MPIO. CANALETE Y LAS CORDOBAS,Rural,Sin Informacion de Barrios,El Cocuelo- Guaimaral- La Mora- Leticia- Lo...
3673,2.855,68,SANTANDER,68298,GAMBITA,6.83,683.037,RÍO PIESCUESTANO,RÍO EL VALLE Y RÍO (PERMANENTE),MPIO. PAIPA Y SOTAQUIRÁ,MUNICIPIO CHITARAQUE,Rural,Sin Informacion de Barrios,Avendaños- Cabecera Municipal- Chinata- El C...
3674,2.509,52,NARIÑO,52079,BARBACOAS,5.28,528.078,RÍO GUIZA,ECUADOR Y MPIO. RICAURTE,RÍO GUIZA,MPIO. SAN ANDRES DE TUMACO,Rural,Sin Informacion de Barrios,Sin Informacion de Veredas
3675,3.413,81,ARAUCA,81001,ARAUCA,8.1,810.008,CAÑO AGUA VERDE,M. PUERTO RONDON,M. CRAVO NORTE,M. ARAUQUITA,Rural,Sin Informacion de Barrios,Cinaruco- Cinaruco- La Pastora- La Pastora- ...
3676,1.2,19,CAUCA,19212,CORINTO,1.915,191.568,MUNICIPIO MIRANDA,MUNICIPIO TORIBÍO,CORDILLERA CENTRAL,VÍA MIRANDA-TORIBÍO,Rural,Sin Informacion de Barrios,La Esperanza- La Playa
3677,245,5,ANTIOQUIA,5360,ITAGÜI,554,55.412,CL 55- 56- 62 Y 63,CL 38- 37B Y 31,KR 49 S,KR 57,Urbano,Araucaria- Artex- Asturias- Centro- El Rosario...,Sin Informacion de Veredas
3678,2.125,47,MAGDALENA,47707,SANTA ANA,4.74,474.027,MPIO. PLATO Y NVA. GRANADA,MPIO. MOMPOS Y TALAIGA NUEVO,VÍA LA GLORIA-PIJIÑO DEL CARMEN,M. SANTA B. DE PINTO,Rural,Sin Informacion de Barrios,LOS MANGOS
3679,2.988,68,SANTANDER,68081,BARRANCABERMEJA,6.87,687.037,MUNICIPIO PUERTO WILCHES,LIMITE URBANO- RIO,MUNICIPIO PUERTO WILCHES,RIO MAGDALENA,Rural,Sin Informacion de Barrios,La Cabaña- La Felicidad- La Rompida N- 2- Pu...
3680,1.166,19,CAUCA,19780,SUAREZ,1.905,190.587,VÍA LA BETULIA-SUÁREZ-SAN IGNACIO,MUNICIPIO MORALES,M. B. AIRES Y MORALES,VÍA LA BETULIA-LOMA EL DIVISO,Rural,Sin Informacion de Barrios,Los Cafés



➡️ 3 filas aleatorias (.sample(3)):


...


➡️ Filas 2 a 5, solo la segunda columna (.iloc[1:5, [1]]):


,codigo_departamento
1,25
2,5
3,88
4,41



➡️ Número de filas (len()): 3681

➡️ Filas y columnas (.shape): (3681, 14)

➡️ Nombre de las columnas (.columns):
['noid', 'codigo_departamento', 'nombre_departamento', 'codigo_municipio', 'nombre_municipio', 'zona_postal', 'codigo_postal', 'limite_norte', 'limite_sur', 'limite_este', 'limite_oeste', 'tipo', 'barrios_contenidos_en_el', 'veredas_contenidas_en_el']

➡️ Tipo de dato de cada columna (.dtypes):


,0
noid,object
codigo_departamento,object
nombre_departamento,object
codigo_municipio,object
nombre_municipio,object
zona_postal,object
codigo_postal,object
limite_norte,object
limite_sur,object
limite_este,object



➡️ Información general (.info()):
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3681 entries, 0 to 3680
Data columns (total 14 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   noid                      3681 non-null   object
 1   codigo_departamento       3681 non-null   object
 2   nombre_departamento       3681 non-null   object
 3   codigo_municipio          3681 non-null   object
 4   nombre_municipio          3681 non-null   object
 5   zona_postal               3681 non-null   object
 6   codigo_postal             3681 non-null   object
 7   limite_norte              3681 non-null   object
 8   limite_sur                3680 non-null   object
 9   limite_este               3680 non-null   object
 10  limite_oeste              3679 non-null   object
 11  tipo                      3681 non-null   object
 12  barrios_contenidos_en_el  3681 non-null   object
 13  veredas_contenidas_en_el  3681 non-null   o

---
# 5️. Otra fuente de datos económicos mediante API: Frankfurter (Banco Central Europeo)

**Frankfurter** (https://frankfurter.dev) es una API pública y gratuita, que no requiere llave de acceso (`API key`),
con las tasas de cambio de referencia publicadas por el Banco Central Europeo desde 1999.

Vamos a traer la serie histórica 2010-2023 del dólar (USD) frente al euro (EUR), la libra esterlina (GBP) y el yen
japonés (JPY).

In [20]:
url_frankfurter = "https://api.frankfurter.app/2010-01-01..2023-12-31?from=USD&to=EUR,GBP,JPY"

response = requests.get(url_frankfurter)
data_fx = response.json()

# La API devuelve un diccionario {fecha: {moneda: tasa}}; lo convertimos a un DataFrame "largo"
df_fx = pd.DataFrame(data_fx["rates"]).T
df_fx.index.name = "fecha"
df_fx = df_fx.reset_index()

df_fx.head()

,fecha,EUR,GBP,JPY
0,2009-12-31,0.69,0.62,92.43
1,2010-01-04,0.69,0.62,92.86
2,2010-01-05,0.69,0.62,91.77
3,2010-01-06,0.70,0.63,92.47
4,2010-01-07,0.70,0.63,93.33


In [21]:
explorar_dataset(df_fx, "Tasas de cambio USD/EUR-GBP-JPY 2010-2023 (Frankfurter API)")


📦 EXPLORACIÓN DE: Tasas de cambio USD/EUR-GBP-JPY 2010-2023 (Frankfurter API)

➡️ Primeras 5 filas (.head()):


,fecha,EUR,GBP,JPY
0,2009-12-31,0.69,0.62,92.43
1,2010-01-04,0.69,0.62,92.86
2,2010-01-05,0.69,0.62,91.77
3,2010-01-06,0.70,0.63,92.47
4,2010-01-07,0.70,0.63,93.33



➡️ Últimas 10 filas (.tail(10)):


,fecha,EUR,GBP,JPY
3577,2023-12-14,0.92,0.79,141.77
3578,2023-12-15,0.91,0.78,141.74
3579,2023-12-18,0.92,0.79,142.84
3580,2023-12-19,0.91,0.79,144.32
3581,2023-12-20,0.91,0.79,143.57
3582,2023-12-21,0.91,0.79,142.76
3583,2023-12-22,0.91,0.79,142.12
3584,2023-12-27,0.90,0.78,142.62
3585,2023-12-28,0.90,0.78,140.88
3586,2023-12-29,0.90,0.79,141.48



➡️ 3 filas aleatorias (.sample(3)):


...


➡️ Filas 2 a 5, solo la segunda columna (.iloc[1:5, [1]]):


,EUR
1,0.69
2,0.69
3,0.70
4,0.70



➡️ Número de filas (len()): 3587

➡️ Filas y columnas (.shape): (3587, 4)

➡️ Nombre de las columnas (.columns):
['fecha', 'EUR', 'GBP', 'JPY']

➡️ Tipo de dato de cada columna (.dtypes):


,0
fecha,object
EUR,float64
GBP,float64
JPY,float64



➡️ Información general (.info()):
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3587 entries, 0 to 3586
Data columns (total 4 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   fecha   3587 non-null   object 
 1   EUR     3587 non-null   float64
 2   GBP     3587 non-null   float64
 3   JPY     3587 non-null   float64
dtypes: float64(3), object(1)
memory usage: 112.2+ KB


---
# ✅ Notas finales de entrega

- Renombrar este archivo como: **`Taller_Importacion_Datos_PenaGonzalezMendoza.ipynb`**
- Subir el notebook a GitHub (no se reciben talleres en Google Colab).
- Enviar el link del repositorio de GitHub en el formulario: https://forms.cloud.microsoft/r/gzNKUEahaK
- Plazo: 27 de agosto de 2026, 11:59 p.m.